# Soup Lab Offload28 Compare

Personal post-hackathon experiment on Kaggle T4. Compare the final slot-view cache with 20 versus 28 activation layers offloaded.
Both snapshots come from https://github.com/kok-o/soup-kernel-lab and are pinned below.

The only runtime difference is `patch_selective_offload(layers=20)` versus `layers=28`.
Each variant runs its patched test suite, the original 2+2 full cold ABBA and all ten original checks.
No performance claim is made until the raw results are checked. Historical 61.40 s /729 MiB is a reference only.

Downloads: `offload-comparison/COMPARISON.json`, `COMPARISON_INPUTS.json`, and each `*-raw-evidence.zip`.
No Brev resources or old competition notebook are used.


In [ ]:
import os, subprocess, sys
from pathlib import Path
CONTROL = '60e973d310ad4b25ecf2eb3d18df284f7a563fcc'
CANDIDATE = '06dc4b1957fbe1e23b4b32fa420d3205890fa553'
REPOSITORY = 'https://github.com/kok-o/soup-kernel-lab.git'
DRIVER = Path('/kaggle/working/offload-comparison-driver')
OUT = Path('/kaggle/working/offload-comparison')
def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
assert not DRIVER.exists() and not OUT.exists(), 'Use a fresh session'
run(['git', 'init', str(DRIVER)])
run(['git', '-C', str(DRIVER), 'config', 'core.autocrlf', 'false'])
run(['git', '-C', str(DRIVER), 'config', 'core.eol', 'lf'])
run(['git', '-C', str(DRIVER), 'remote', 'add', 'origin', REPOSITORY])
run(['git', '-C', str(DRIVER), 'fetch', '--depth', '1', 'origin', CANDIDATE])
run(['git', '-C', str(DRIVER), 'checkout', '--detach', 'FETCH_HEAD'])
assert subprocess.check_output(['git', '-C', str(DRIVER), 'rev-parse', 'HEAD'], text=True).strip() == CANDIDATE
gpus = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
print(gpus)
assert 'T4' in gpus, 'Use Tesla T4'


In [ ]:
# Installation is outside the official timed child processes.
run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'xformers', 'flash-attn', 'liger-kernel', 'unsloth', 'unsloth-zoo'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.11.0', '--index-url', 'https://download.pytorch.org/whl/cu128'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'triton==3.6.0', 'soup-cli[train]==0.77.0', 'transformers==5.16.1', 'peft==0.20.0', 'trl==0.29.1', 'bitsandbytes==0.50.2', 'accelerate==1.14.0', 'datasets==4.8.5', 'pytest'])


In [ ]:
# No torch import/context in this notebook process. GPU work is in ended children.
run([sys.executable, 'research/run_offload_comparison.py', '--control', CONTROL,
     '--candidate', CANDIDATE, '--out', str(OUT)], cwd=DRIVER)


In [ ]:
import json
report = json.loads((OUT / 'COMPARISON.json').read_text())
for name, result in report['results'].items():
    print(name, result['commit'], 'all_checks:', result['all_checks'])
    print(json.dumps(result.get('summary', result), indent=2))
print(report['limitations'])
